# 06 · Track, compare and promote

**What you will do**
1. Register the models in the project's **MLflow model registry**, with aliases (`champion`, `challenger`, `edge`).
2. Build an **AI Pipeline** that scores the current model and a challenger on the same held-out notes, and promotes
   the challenger only if it is better.
3. Run it from this notebook and see the run in the dashboard.

**GPU:** none (the pipeline calls the endpoint from notebook 04). **Time:** about 15 minutes.

| | Dev environment | On OpenShift AI |
|---|---|---|
| Tracking | Stand up your own MLflow | MLflow per project, with a model registry |
| Promotion | Someone compares numbers | A pipeline compares and moves the `champion` alias |
| Audit | Ad hoc | Every run, scorecard and decision recorded |

In [ ]:
# Settings used by every notebook. Change them here if your project differs.
import os, json, time
from pathlib import Path

NAMESPACE  = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()   # this project
WS         = Path("/opt/app-root/src/workspace")   # the project volume, as mounted in this workbench
PVC        = "workspace"                           # its name; training and serving pods mount it at /workspace
REPO       = Path.cwd().parent                     # this repository (data/, sdg_flows/)
BASE_MODEL = "ibm-granite/granite-4.1-3b"
EXPERIMENT = "sentinel7"                           # MLflow experiment
NODE_SELECTOR = {}   # only with ReadWriteOnce storage: keep every pod that mounts the volume on one node
TRAINING_IMAGE = ("registry.redhat.io/rhoai/odh-th-torch-cuda-py312-rhel9@sha256:"   # OpenShift AI 3.5 Training Hub image
                  "884aeeb039f9592252fe0c518acf31803a208757d4fa6500fe79884a022ea52d")

os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", EXPERIMENT)
print("project:", NAMESPACE, "| volume:", WS, "| MLflow:", os.environ.get("MLFLOW_TRACKING_URI", "not enabled"))

In [ ]:
# Create OpenShift AI resources with the workbench's own identity (no token to paste).
import yaml
from kubernetes import client as k8s, config, dynamic
config.load_incluster_config()
dyn = dynamic.DynamicClient(k8s.ApiClient())

def apply(manifest: str):
    """Create or update every object in a YAML string, in this project."""
    for obj in yaml.safe_load_all(manifest):
        res = dyn.resources.get(api_version=obj["apiVersion"], kind=obj["kind"])
        name = obj["metadata"]["name"]
        try:
            dyn.create(res, body=obj, namespace=NAMESPACE); print("created", obj["kind"], name)
        except k8s.ApiException as e:
            if e.status != 409: raise
            dyn.patch(res, body=obj, name=name, namespace=NAMESPACE, content_type="application/merge-patch+json")
            print("updated", obj["kind"], name)

def run_job(manifest: str, show=None):
    """(Re)start a Kubernetes Job and stream its log. `show` keeps only lines containing one of these words."""
    obj = yaml.safe_load(manifest); name = obj["metadata"]["name"]
    batch, core = k8s.BatchV1Api(), k8s.CoreV1Api()
    try:
        batch.delete_namespaced_job(name, NAMESPACE, propagation_policy="Foreground")
        while batch.list_namespaced_job(NAMESPACE, field_selector=f"metadata.name={name}").items: time.sleep(3)
    except k8s.ApiException: pass
    if NODE_SELECTOR: obj["spec"]["template"]["spec"]["nodeSelector"] = NODE_SELECTOR
    batch.create_namespaced_job(NAMESPACE, obj)
    while True:
        pods = core.list_namespaced_pod(NAMESPACE, label_selector=f"job-name={name}").items
        if pods and pods[0].status.phase in ("Running", "Succeeded", "Failed"): break
        time.sleep(5)
    for raw in core.read_namespaced_pod_log(pods[0].metadata.name, NAMESPACE, follow=True, _preload_content=False):
        for line in raw.decode(errors="replace").splitlines():
            if show is None or any(w in line for w in show): print(line, flush=True)
    s = batch.read_namespaced_job(name, NAMESPACE).status
    print(f"\nJob {name}: {'succeeded' if s.succeeded else 'FAILED - see the log above'}")

def wait_ready(kind, api_version, name, timeout=1800):
    res = dyn.resources.get(api_version=api_version, kind=kind); t0 = time.time()
    while time.time() - t0 < timeout:
        conds = (dyn.get(res, name=name, namespace=NAMESPACE).to_dict().get("status") or {}).get("conditions", [])
        if any(c["type"] == "Ready" and c["status"] == "True" for c in conds):
            return print(f"{name} is ready ({time.time() - t0:.0f} s)")
        time.sleep(10)
    raise TimeoutError(name)

## 1 · Register the models

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
mc, REG = MlflowClient(), "sentinel7-edge-copilot"
try: mc.create_registered_model(REG, description="Radio note -> BD-SCR JSON report. Granite 4.1-3B + LoRA, Training Hub.")
except Exception: pass

def register(source, description):
    for v in mc.search_model_versions(f"name = '{REG}'"):
        if v.source == source: return v.version
    return mc.create_model_version(REG, source=source, description=description).version

versions = {}
for name, desc in [("edgecopilot-sft", "SFT on 2,400 notes"), ("edgecopilot-sft-v2", "SFT + 600 radio-style notes")]:
    if (WS / "checkpoints" / name / "adapter_model.safetensors").exists():
        versions[name] = register(f"pvc://{PVC}/checkpoints/{name}", desc)
if (WS / "export/edgecopilot/edgecopilot-Q4_K_M.gguf").exists():
    versions["edge"] = register(f"pvc://{PVC}/export/edgecopilot/edgecopilot-Q4_K_M.gguf", "4-bit GGUF for llama.cpp")

mc.set_registered_model_alias(REG, "champion", versions["edgecopilot-sft"])
if "edgecopilot-sft-v2" in versions: mc.set_registered_model_alias(REG, "challenger", versions["edgecopilot-sft-v2"])
if "edge" in versions: mc.set_registered_model_alias(REG, "edge", versions["edge"])
print(versions, mc.get_registered_model(REG).aliases)

## 2 · The pipeline server

Pipelines run on the project's pipeline server. Create it once in the dashboard (Develop & train > Pipelines >
Configure pipeline server), or with the cell below (a small demo setup with its own storage).

In [ ]:
apply("""
apiVersion: datasciencepipelinesapplications.opendatahub.io/v1
kind: DataSciencePipelinesApplication
metadata: {name: dspa}
spec:
  dspVersion: v2
  apiServer: {deploy: true, enableSamplePipeline: false}
  database: {mariaDB: {deploy: true, pipelineDBName: mlpipeline, username: mlpipeline, pvcSize: 2Gi}}
  objectStorage:
    minio: {deploy: true, bucket: mlpipeline, pvcSize: 5Gi,
            image: "quay.io/opendatahub/minio:RELEASE.2019-08-14T20-37-41Z-license-compliance"}
""")

## 3 · The pipeline: evaluate both, compare, promote

Four steps, each a small Python function: load held-out notes → evaluate the current model and the challenger
(in parallel) → quality gate → record the decision in MLflow and move the alias if the challenger wins.

In [ ]:
%pip install -q kfp kfp-kubernetes

In [ ]:
from typing import NamedTuple
from kfp import dsl, compiler, kubernetes
from kfp.dsl import Dataset, Input, Output

IMAGE = "registry.access.redhat.com/ubi9/python-312:latest"

@dsl.component(base_image=IMAGE)
def load_notes(n: int, notes: Output[Dataset]):
    lines = open("/workspace/data/edgecopilot/test.jsonl").read().splitlines()[:n]
    open(notes.path, "w").write("\n".join(lines))

@dsl.component(base_image=IMAGE)
def evaluate(model: str, url: str, notes: Input[Dataset]) -> NamedTuple("Card", [("exact", float), ("invented", float)]):
    import json, ssl, sys, urllib.request
    sys.path.insert(0, "/workspace/code")
    from report_check import scorecard
    ctx = ssl.create_default_context(); ctx.check_hostname, ctx.verify_mode = False, ssl.CERT_NONE
    rows = [json.loads(l) for l in open(notes.path)]
    def ask(r):
        body = json.dumps({"model": model, "messages": r["messages"][:2], "temperature": 0, "max_tokens": 700}).encode()
        req = urllib.request.Request(f"{url}/chat/completions", body, {"Content-Type": "application/json"})
        return json.load(urllib.request.urlopen(req, context=ctx, timeout=300))["choices"][0]["message"]["content"]
    from collections import namedtuple
    card = scorecard([ask(r) for r in rows], [json.loads(r["messages"][2]["content"]) for r in rows])
    print(model, card)
    return namedtuple("Card", ["exact", "invented"])(card["Exact report %"], card["Invented a field %"])

@dsl.component(base_image=IMAGE)
def gate(cur_exact: float, cur_inv: float, new_exact: float, new_inv: float) -> str:
    """Promote only if strictly better on exact reports and no worse on invented fields."""
    decision = "promote" if new_exact > cur_exact and new_inv <= cur_inv else "keep"
    print(f"current {cur_exact}% / challenger {new_exact}% -> {decision}")
    return decision

@dsl.component(base_image=IMAGE, packages_to_install=["mlflow-skinny==3.14.0", "kubernetes"])
def record(decision: str, registered_model: str, challenger_version: str, cur_exact: float, new_exact: float):
    import mlflow
    from mlflow.tracking import MlflowClient
    with mlflow.start_run(run_name="eval-gate-promote"):
        mlflow.set_tag("decision", decision)
        mlflow.log_metrics({"current_exact_pct": cur_exact, "challenger_exact_pct": new_exact})
    if decision == "promote":
        MlflowClient().set_registered_model_alias(registered_model, "champion", challenger_version)

@dsl.pipeline(name="sentinel7-eval-gate-promote")
def eval_gate_promote(url: str, current: str, challenger: str, challenger_version: str,
                      n_notes: int = 50, registered_model: str = "sentinel7-edge-copilot"):
    notes = load_notes(n=n_notes)
    a = evaluate(model=current, url=url, notes=notes.outputs["notes"]).set_display_name("Evaluate current model")
    b = evaluate(model=challenger, url=url, notes=notes.outputs["notes"]).set_display_name("Evaluate challenger")
    g = gate(cur_exact=a.outputs["exact"], cur_inv=a.outputs["invented"],
             new_exact=b.outputs["exact"], new_inv=b.outputs["invented"]).set_display_name("Quality gate")
    r = record(decision=g.output, registered_model=registered_model, challenger_version=challenger_version,
               cur_exact=a.outputs["exact"], new_exact=b.outputs["exact"]).set_display_name("Record and promote")
    for t in (notes, a, b):
        kubernetes.mount_pvc(t, pvc_name=PVC, mount_path="/workspace")
        for k, v in NODE_SELECTOR.items(): kubernetes.add_node_selector(t, label_key=k, label_value=v)
    for k, v in {"MLFLOW_TRACKING_URI": os.environ.get("MLFLOW_TRACKING_URI", ""), "MLFLOW_EXPERIMENT_NAME": EXPERIMENT,
                 "MLFLOW_TRACKING_AUTH": "kubernetes-namespaced", "MLFLOW_K8S_INTEGRATION": "true"}.items():
        r.set_env_variable(k, v)
    for t in (notes, a, b, g, r): t.set_caching_options(False)

compiler.Compiler().compile(eval_gate_promote, "eval-gate-promote.yaml")
print("compiled eval-gate-promote.yaml")

## 4 · Run it

In [ ]:
import kfp
route = dyn.get(dyn.resources.get(api_version="route.openshift.io/v1", kind="Route"),
                name="ds-pipeline-dspa", namespace=NAMESPACE).to_dict()["spec"]["host"]
token = open("/var/run/secrets/kubernetes.io/serviceaccount/token").read()
kfp_client = kfp.Client(host=f"https://{route}", existing_token=token, ssl_ca_cert=None)

run = kfp_client.create_run_from_pipeline_package("eval-gate-promote.yaml", arguments={
    "url": f"https://granite-41-3b-kserve-workload-svc.{NAMESPACE}.svc:8000/v1",
    "current": "edgecopilot-sft", "challenger": "edgecopilot-sft-v2",
    "challenger_version": str(versions.get("edgecopilot-sft-v2", ""))}, experiment_name=EXPERIMENT)
print("run:", run.run_id, "- follow it in the dashboard: Develop & train > Pipelines > Runs")
kfp_client.wait_for_run_completion(run.run_id, timeout=1800)
print("aliases now:", mc.get_registered_model(REG).aliases)

## What we measured

The same gate on our models: SFT v2 (current, 100% on 50 notes) against v1 + DPO (challenger, 100%). The challenger
was not strictly better, so the pipeline **kept the current model** and recorded why.

## Where to go next

- **EvalHub** (generally available in OpenShift AI 3.5) adds risk and compliance evaluation with auditable reports.
- Red Hat publishes ready-made fine-tuning pipelines (train → evaluate → register) you can start from the dashboard.
- The scenario is fictional; swap in your own data, checker and rules, and every notebook works the same way.